# Module 4.2: Natural Gas Storage & Balance Modeling

## Learning Objectives
By completing this notebook, you will:
1. Build storage-centric natural gas balance models
2. Incorporate weather data and degree days into demand forecasts
3. Analyze injection/withdrawal patterns and their price impact
4. Generate LLM-enhanced forward balance projections

## Prerequisites
- Completion of Module 4.1 (Crude Fundamentals)
- Understanding of natural gas markets and storage dynamics
- Familiarity with seasonal commodity patterns

## Estimated Time: 90 minutes

---

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
learning_objectives([
    "Completion of Module 4.1 (Crude Fundamentals)",
    "Understanding of natural gas markets and storage dynamics",
    "Familiarity with seasonal commodity patterns"
])

## Setup & Imports

In [ ]:
# Purpose: Import libraries and configure environment
# Key Concept: Natural gas analysis requires weather and seasonality tools

import os
import json
import requests
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Literal
from dataclasses import dataclass, asdict
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from langchain_anthropic import ChatAnthropic
from langchain.prompts import ChatPromptTemplate
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# Set display options
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
np.random.seed(42)

print("✓ Imports complete")

## 1. Understanding Natural Gas Fundamentals

### Why Natural Gas is Different from Crude Oil

Natural gas markets are **storage-driven** rather than flow-driven:

**Key Differences:**

| Aspect | Crude Oil | Natural Gas |
|--------|-----------|-------------|
| Storage | Strategic, commercial | Seasonal buffer |
| Seasonality | Moderate | Extreme (heating/cooling) |
| Transportation | Global (tankers) | Regional (pipelines) |
| Pricing | Global benchmarks | Regional hubs |
| Weather Sensitivity | Low | Very high |

### Storage Cycle

```
Apr-Oct: INJECTION SEASON
├── Low demand (shoulder months)
├── Build inventory for winter
└── Target: Fill storage by Nov 1

Nov-Mar: WITHDRAWAL SEASON  
├── High heating demand
├── Draw down inventory
└── Low point: End of March
```

### Critical Metrics

1. **Storage Level vs 5-Year Average**: Primary market signal
2. **Weekly Injection/Withdrawal**: Indicates current balance
3. **Heating Degree Days (HDD)**: Winter demand driver
4. **Cooling Degree Days (CDD)**: Summer demand driver

## 2. Natural Gas Data Models

In [ ]:
# Purpose: Define data structures for natural gas fundamentals
# Key Concept: Storage-centric model with weather integration

class NatGasStorage(BaseModel):
    """Natural gas storage data."""
    
    report_date: str = Field(description="Report date (YYYY-MM-DD)")
    level: float = Field(description="Storage level in billion cubic feet (Bcf)")
    weekly_change: float = Field(description="Net injection (+) or withdrawal (-)")
    year_ago_level: float = Field(description="Storage level same week last year")
    five_year_avg: float = Field(description="5-year average for this week")
    vs_year_ago: float = Field(description="Difference from year ago (Bcf)")
    vs_year_ago_pct: float = Field(description="Percent vs year ago")
    vs_5yr_avg: float = Field(description="Difference from 5-year average (Bcf)")
    vs_5yr_avg_pct: float = Field(description="Percent vs 5-year average")


class WeatherData(BaseModel):
    """Weather data for demand modeling."""
    
    period: str = Field(description="Time period")
    heating_degree_days: float = Field(description="Total HDD for period")
    cooling_degree_days: float = Field(description="Total CDD for period")
    vs_normal_hdd: float = Field(description="HDD vs 30-year normal")
    vs_normal_cdd: float = Field(description="CDD vs 30-year normal")
    outlook: str = Field(description="Weather outlook (warmer/colder than normal)")


class NatGasSupplyDemand(BaseModel):
    """Natural gas supply and demand data."""
    
    production: float = Field(description="Dry natural gas production (Bcf/d)")
    production_change: Optional[float] = Field(default=None, description="Change from prior period")
    imports: float = Field(description="Imports (Bcf/d)")
    lng_exports: float = Field(description="LNG exports (Bcf/d)")
    pipeline_exports: float = Field(description="Pipeline exports (Bcf/d)")
    residential_commercial: float = Field(description="Res/comm consumption (Bcf/d)")
    industrial: float = Field(description="Industrial consumption (Bcf/d)")
    power_generation: float = Field(description="Power sector consumption (Bcf/d)")


class NatGasFundamentals(BaseModel):
    """Complete natural gas fundamental snapshot."""
    
    report_date: str
    storage: NatGasStorage
    supply_demand: NatGasSupplyDemand
    weather: Optional[WeatherData] = None
    qualitative_factors: List[str]
    outlook: str


print("✓ Natural gas data models defined")

## 3. Extracting Natural Gas Fundamentals

In [ ]:
# Purpose: Build LLM extractor for EIA natural gas storage reports
# Key Concept: Specialized extraction for storage-centric data

class NatGasExtractor:
    """Extract natural gas fundamentals from EIA reports."""
    
    def __init__(self, api_key: Optional[str] = None):
        self.llm = ChatAnthropic(
            model="claude-3-5-sonnet-20241022",
            api_key=api_key or os.environ.get("ANTHROPIC_API_KEY"),
            temperature=0
        )
        self.parser = PydanticOutputParser(pydantic_object=NatGasFundamentals)
        
    def extract(self, report_text: str) -> NatGasFundamentals:
        """
        Extract natural gas fundamentals from EIA storage report.
        
        Parameters
        ----------
        report_text : str
            EIA Natural Gas Weekly Update or Storage Report text
            
        Returns
        -------
        NatGasFundamentals
            Structured fundamental data
        """
        prompt = ChatPromptTemplate.from_messages([
            ("system", """You are an expert natural gas market analyst extracting data 
from EIA natural gas reports.

Focus on:
- Storage levels and weekly injections/withdrawals
- Comparisons to year-ago and 5-year average
- Supply components (production, imports)
- Demand components (residential, industrial, power gen, exports)
- Weather data (heating/cooling degree days)
- Forward outlook and seasonal expectations

Be precise with numbers and units (Bcf, Bcf/d).

{format_instructions}"""),
            ("user", "Extract natural gas fundamentals from this report:\n\n{report_text}")
        ])
        
        chain = prompt | self.llm
        response = chain.invoke({
            "report_text": report_text,
            "format_instructions": self.parser.get_format_instructions()
        })
        
        return self.parser.parse(response.content)


extractor = NatGasExtractor()
print("✓ Natural gas extractor initialized")

Test with a sample EIA Natural Gas Storage Report.

In [ ]:
# Purpose: Test extraction on sample EIA natural gas storage report
# Key Concept: Real-world natural gas fundamental data extraction

sample_gas_report = """
EIA Natural Gas Weekly Update - December 28, 2023
Storage Report for Week Ending December 22, 2023

STORAGE:
Working gas in storage was 3,522 Bcf as of Friday, December 22, 2023, according to EIA 
estimates. This represents a net withdrawal of 112 Bcf from the previous week.

Total working gas is 281 Bcf (8.7%) higher than the year-ago level and 325 Bcf (10.2%) 
higher than the five-year average of 3,197 Bcf.

SUPPLY & DEMAND:
Dry natural gas production averaged 104.5 Bcf/d during the report week, up slightly from 
104.2 Bcf/d the previous week. Production is running 3% above year-ago levels.

Total U.S. consumption of natural gas averaged 126.8 Bcf/d, an increase of 8.2 Bcf/d 
(6.9%) compared with the previous report week.

- Residential and commercial consumption: 52.3 Bcf/d
- Industrial consumption: 26.8 Bcf/d  
- Power generation: 33.5 Bcf/d
- LNG exports: 13.8 Bcf/d
- Pipeline exports to Mexico: 6.4 Bcf/d

WEATHER:
The report week recorded 178 heating degree days (HDDs), 18% above normal and 12% above 
the same week last year. Cold weather across the Midwest and Northeast drove increased 
residential heating demand.

OUTLOOK:
The National Weather Service forecasts below-normal temperatures for much of the eastern 
U.S. through mid-January, which is expected to support continued strong withdrawals. 
However, storage levels remain well above the 5-year average, providing a comfortable 
cushion for winter demand.

KEY FACTORS:
- LNG export capacity reached record highs as new facility ramps up
- Pipeline maintenance in Permian reduced associated gas production
- Coal-to-gas switching in power sector due to low gas prices
"""

# Extract fundamentals
gas_fundamentals = extractor.extract(sample_gas_report)

# Display results
print("=== STORAGE DATA ===")
print(f"Current Level: {gas_fundamentals.storage.level:.0f} Bcf")
print(f"Weekly Change: {gas_fundamentals.storage.weekly_change:+.0f} Bcf (withdrawal)")
print(f"vs Year Ago: {gas_fundamentals.storage.vs_year_ago_pct:+.1f}%")
print(f"vs 5-Yr Avg: {gas_fundamentals.storage.vs_5yr_avg_pct:+.1f}%")

print("\n=== SUPPLY ===")
print(f"Production: {gas_fundamentals.supply_demand.production:.1f} Bcf/d")
print(f"Imports: {gas_fundamentals.supply_demand.imports:.1f} Bcf/d")

print("\n=== DEMAND ===")
print(f"Residential/Commercial: {gas_fundamentals.supply_demand.residential_commercial:.1f} Bcf/d")
print(f"Industrial: {gas_fundamentals.supply_demand.industrial:.1f} Bcf/d")
print(f"Power Generation: {gas_fundamentals.supply_demand.power_generation:.1f} Bcf/d")
print(f"LNG Exports: {gas_fundamentals.supply_demand.lng_exports:.1f} Bcf/d")
print(f"Pipeline Exports: {gas_fundamentals.supply_demand.pipeline_exports:.1f} Bcf/d")

if gas_fundamentals.weather:
    print("\n=== WEATHER ===")
    print(f"HDDs: {gas_fundamentals.weather.heating_degree_days:.0f}")
    print(f"vs Normal: {gas_fundamentals.weather.vs_normal_hdd:+.1f}%")

## 4. Storage Balance Analysis

Calculate whether current withdrawal rates are consistent with typical seasonal patterns.

In [ ]:
# Purpose: Analyze storage balance and seasonal patterns
# Key Concept: Storage trajectory relative to seasonal norms

@dataclass
class StorageBalance:
    """Natural gas storage balance analysis."""
    
    current_level: float
    vs_5yr_avg_bcf: float
    vs_5yr_avg_pct: float
    weekly_change: float
    season: Literal["injection", "withdrawal", "shoulder"]
    implied_weeks_supply: float
    trajectory_assessment: str
    market_implication: str


def analyze_storage_balance(fundamentals: NatGasFundamentals,
                           avg_weekly_withdrawal: float = 150.0) -> StorageBalance:
    """
    Analyze natural gas storage balance.
    
    Parameters
    ----------
    fundamentals : NatGasFundamentals
        Current fundamental data
    avg_weekly_withdrawal : float
        Average withdrawal rate during winter (Bcf/week)
        
    Returns
    -------
    StorageBalance
        Storage balance analysis
    """
    storage = fundamentals.storage
    
    # Determine season based on withdrawal/injection
    if storage.weekly_change < -20:
        season = "withdrawal"
    elif storage.weekly_change > 20:
        season = "injection"
    else:
        season = "shoulder"
    
    # Calculate implied weeks of supply
    if season == "withdrawal" and storage.weekly_change < 0:
        implied_weeks = storage.level / abs(storage.weekly_change)
    else:
        implied_weeks = storage.level / avg_weekly_withdrawal
    
    # Assess trajectory
    if storage.vs_5yr_avg_pct > 10:
        trajectory = "WELL ABOVE AVERAGE: Ample supply cushion"
        market_impact = "BEARISH: Oversupplied, downward price pressure"
    elif storage.vs_5yr_avg_pct > 0:
        trajectory = "ABOVE AVERAGE: Adequate supply"
        market_impact = "NEUTRAL TO BEARISH: Slight oversupply"
    elif storage.vs_5yr_avg_pct > -10:
        trajectory = "BELOW AVERAGE: Tighter than normal"
        market_impact = "NEUTRAL TO BULLISH: Moderate tightness"
    else:
        trajectory = "WELL BELOW AVERAGE: Critically tight"
        market_impact = "BULLISH: Supply concerns, upward price pressure"
    
    return StorageBalance(
        current_level=storage.level,
        vs_5yr_avg_bcf=storage.vs_5yr_avg,
        vs_5yr_avg_pct=storage.vs_5yr_avg_pct,
        weekly_change=storage.weekly_change,
        season=season,
        implied_weeks_supply=implied_weeks,
        trajectory_assessment=trajectory,
        market_implication=market_impact
    )


# Analyze storage balance
storage_balance = analyze_storage_balance(gas_fundamentals)

print("=== STORAGE BALANCE ANALYSIS ===")
print(f"Current Level: {storage_balance.current_level:.0f} Bcf")
print(f"vs 5-Year Average: {storage_balance.vs_5yr_avg_pct:+.1f}% ({storage_balance.vs_5yr_avg_bcf:+.0f} Bcf)")
print(f"Weekly Change: {storage_balance.weekly_change:+.0f} Bcf")
print(f"Season: {storage_balance.season.upper()}")
print(f"Implied Weeks Supply: {storage_balance.implied_weeks_supply:.1f} weeks")
print(f"\nTrajectory: {storage_balance.trajectory_assessment}")
print(f"Market Implication: {storage_balance.market_implication}")

## 5. Weather-Adjusted Demand Modeling

Natural gas demand is highly weather-sensitive. We'll model the relationship between degree days and consumption.

In [ ]:
# Purpose: Model weather-driven natural gas demand
# Key Concept: HDD/CDD correlation with residential/commercial demand

def generate_weather_demand_data(num_weeks: int = 52) -> pd.DataFrame:
    """
    Generate synthetic weather and demand data for demonstration.
    In production, use actual NOAA weather data and EIA consumption data.
    
    Parameters
    ----------
    num_weeks : int
        Number of weeks to generate
        
    Returns
    -------
    pd.DataFrame
        Weather and demand time series
    """
    dates = pd.date_range(end=datetime.now(), periods=num_weeks, freq='W')
    
    data = []
    for i, date in enumerate(dates):
        # Model seasonal pattern (winter = high HDD, summer = high CDD)
        week_of_year = date.isocalendar()[1]
        
        # HDD peaks in winter (weeks 1-10, 50-52)
        hdd_seasonal = 200 * np.cos(2 * np.pi * (week_of_year - 2) / 52) + 100
        hdd = max(0, hdd_seasonal + np.random.normal(0, 30))
        
        # CDD peaks in summer (weeks 25-35)
        cdd_seasonal = 150 * np.cos(2 * np.pi * (week_of_year - 28) / 52) + 50
        cdd = max(0, cdd_seasonal + np.random.normal(0, 20))
        
        # Demand driven by weather + baseline
        baseline_demand = 85  # Bcf/d baseline (industrial + power)
        weather_demand = (hdd * 0.15) + (cdd * 0.08)  # Weather-sensitive demand
        total_demand = baseline_demand + weather_demand + np.random.normal(0, 3)
        
        data.append({
            'date': date,
            'week_of_year': week_of_year,
            'hdd': hdd,
            'cdd': cdd,
            'total_demand': total_demand,
            'baseline_demand': baseline_demand,
            'weather_demand': weather_demand
        })
    
    return pd.DataFrame(data)


# Generate data
weather_demand_data = generate_weather_demand_data()

# Analyze correlation
hdd_corr = weather_demand_data[['hdd', 'total_demand']].corr().iloc[0, 1]
cdd_corr = weather_demand_data[['cdd', 'total_demand']].corr().iloc[0, 1]

print(f"HDD-Demand Correlation: {hdd_corr:.3f}")
print(f"CDD-Demand Correlation: {cdd_corr:.3f}")
print(f"\nSample Data:")
print(weather_demand_data.tail())

In [ ]:
# Purpose: Visualize weather-demand relationship
# Key Concept: Understanding seasonal demand drivers

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Plot 1: Degree Days
ax1 = axes[0]
ax1.fill_between(weather_demand_data['date'], weather_demand_data['hdd'], 
                  alpha=0.5, label='Heating Degree Days', color='blue')
ax1.fill_between(weather_demand_data['date'], weather_demand_data['cdd'], 
                  alpha=0.5, label='Cooling Degree Days', color='red')
ax1.set_ylabel('Degree Days', fontsize=12)
ax1.set_title('Weather Patterns (Degree Days)', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Demand Components
ax2 = axes[1]
ax2.plot(weather_demand_data['date'], weather_demand_data['total_demand'], 
         linewidth=2, label='Total Demand', color='black')
ax2.plot(weather_demand_data['date'], weather_demand_data['baseline_demand'], 
         linewidth=2, linestyle='--', label='Baseline Demand', color='gray')
ax2.fill_between(weather_demand_data['date'], 
                  weather_demand_data['baseline_demand'],
                  weather_demand_data['total_demand'],
                  alpha=0.3, label='Weather-Driven Demand', color='orange')
ax2.set_ylabel('Bcf/d', fontsize=12)
ax2.set_title('Natural Gas Demand', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Seasonal Pattern
ax3 = axes[2]
monthly_avg = weather_demand_data.groupby('week_of_year').agg({
    'total_demand': 'mean',
    'hdd': 'mean',
    'cdd': 'mean'
})
ax3_twin = ax3.twinx()
ax3.plot(monthly_avg.index, monthly_avg['total_demand'], 
         linewidth=2, color='green', marker='o', label='Avg Demand')
ax3_twin.bar(monthly_avg.index, monthly_avg['hdd'], 
             alpha=0.3, color='blue', label='Avg HDD')
ax3_twin.bar(monthly_avg.index, monthly_avg['cdd'], 
             alpha=0.3, color='red', label='Avg CDD')
ax3.set_xlabel('Week of Year', fontsize=12)
ax3.set_ylabel('Demand (Bcf/d)', fontsize=12, color='green')
ax3_twin.set_ylabel('Degree Days', fontsize=12)
ax3.set_title('Seasonal Demand Pattern', fontsize=14, fontweight='bold')
ax3.legend(loc='upper left')
ax3_twin.legend(loc='upper right')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Storage Trajectory Forecasting

Project forward storage levels based on current balance and weather forecasts.

In [ ]:
# Purpose: Forecast storage trajectory using weather and balance data
# Key Concept: Forward-looking supply adequacy assessment

def forecast_storage_trajectory(current_level: float,
                               weeks_forward: int = 12,
                               avg_withdrawal: float = 150.0,
                               weather_adjustment: float = 1.0) -> pd.DataFrame:
    """
    Forecast storage levels based on expected withdrawal rates.
    
    Parameters
    ----------
    current_level : float
        Current storage level (Bcf)
    weeks_forward : int
        Number of weeks to forecast
    avg_withdrawal : float
        Expected average weekly withdrawal (Bcf)
    weather_adjustment : float
        Weather multiplier (>1 = colder, <1 = warmer)
        
    Returns
    -------
    pd.DataFrame
        Forecasted storage trajectory
    """
    dates = pd.date_range(start=datetime.now(), periods=weeks_forward, freq='W')
    
    trajectory = []
    storage = current_level
    
    for i, date in enumerate(dates):
        # Seasonal withdrawal pattern (higher early winter, tapering in spring)
        week_factor = 1.0 + 0.3 * np.cos(2 * np.pi * i / 20)
        weekly_change = -avg_withdrawal * week_factor * weather_adjustment
        weekly_change += np.random.normal(0, 20)  # Add uncertainty
        
        storage += weekly_change
        
        # Calculate vs 5-year average (declining through winter)
        five_year_avg = 3200 - (i * 120)  # Synthetic average trajectory
        vs_avg_pct = ((storage - five_year_avg) / five_year_avg) * 100
        
        trajectory.append({
            'date': date,
            'storage_level': storage,
            'weekly_change': weekly_change,
            'five_year_avg': five_year_avg,
            'vs_avg_pct': vs_avg_pct
        })
    
    return pd.DataFrame(trajectory)


# Generate baseline forecast
baseline_forecast = forecast_storage_trajectory(
    current_level=gas_fundamentals.storage.level,
    avg_withdrawal=abs(gas_fundamentals.storage.weekly_change)
)

# Generate cold weather scenario
cold_scenario = forecast_storage_trajectory(
    current_level=gas_fundamentals.storage.level,
    avg_withdrawal=abs(gas_fundamentals.storage.weekly_change),
    weather_adjustment=1.3  # 30% colder
)

# Generate warm weather scenario
warm_scenario = forecast_storage_trajectory(
    current_level=gas_fundamentals.storage.level,
    avg_withdrawal=abs(gas_fundamentals.storage.weekly_change),
    weather_adjustment=0.7  # 30% warmer
)

print("Storage Trajectory Forecast:")
print(baseline_forecast.head())

In [ ]:
# Purpose: Visualize storage trajectory scenarios
# Key Concept: Range of outcomes based on weather uncertainty

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Plot 1: Storage Level Scenarios
ax1 = axes[0]
ax1.plot(baseline_forecast['date'], baseline_forecast['storage_level'],
         linewidth=2, label='Baseline (Normal Weather)', color='blue')
ax1.plot(cold_scenario['date'], cold_scenario['storage_level'],
         linewidth=2, linestyle='--', label='Cold Scenario (+30%)', color='darkblue')
ax1.plot(warm_scenario['date'], warm_scenario['storage_level'],
         linewidth=2, linestyle='--', label='Warm Scenario (-30%)', color='lightblue')
ax1.plot(baseline_forecast['date'], baseline_forecast['five_year_avg'],
         linewidth=2, linestyle=':', label='5-Year Average', color='gray')
ax1.fill_between(cold_scenario['date'], 
                  cold_scenario['storage_level'],
                  warm_scenario['storage_level'],
                  alpha=0.2, color='blue')
ax1.set_ylabel('Storage Level (Bcf)', fontsize=12)
ax1.set_title('Storage Trajectory Forecast', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Deviation from 5-Year Average
ax2 = axes[1]
ax2.plot(baseline_forecast['date'], baseline_forecast['vs_avg_pct'],
         linewidth=2, label='Baseline', color='green')
ax2.plot(cold_scenario['date'], cold_scenario['vs_avg_pct'],
         linewidth=2, linestyle='--', label='Cold Scenario', color='darkgreen')
ax2.plot(warm_scenario['date'], warm_scenario['vs_avg_pct'],
         linewidth=2, linestyle='--', label='Warm Scenario', color='lightgreen')
ax2.axhline(y=0, color='black', linestyle='--', linewidth=1)
ax2.fill_between(baseline_forecast['date'], 0, baseline_forecast['vs_avg_pct'],
                  where=baseline_forecast['vs_avg_pct']>=0,
                  alpha=0.3, color='red', label='Above Average')
ax2.fill_between(baseline_forecast['date'], 0, baseline_forecast['vs_avg_pct'],
                  where=baseline_forecast['vs_avg_pct']<0,
                  alpha=0.3, color='green', label='Below Average')
ax2.set_xlabel('Date', fontsize=12)
ax2.set_ylabel('% vs 5-Year Avg', fontsize=12)
ax2.set_title('Storage Deviation from Average', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print scenario outcomes
print("\n=== 12-WEEK SCENARIO OUTCOMES ===")
print(f"Baseline End Level: {baseline_forecast['storage_level'].iloc[-1]:.0f} Bcf")
print(f"Cold Scenario End Level: {cold_scenario['storage_level'].iloc[-1]:.0f} Bcf")
print(f"Warm Scenario End Level: {warm_scenario['storage_level'].iloc[-1]:.0f} Bcf")
print(f"Range: {cold_scenario['storage_level'].iloc[-1] - warm_scenario['storage_level'].iloc[-1]:.0f} Bcf")

## 7. LLM-Generated Market Assessment

In [ ]:
# Purpose: Generate comprehensive natural gas market assessment
# Key Concept: Synthesizing fundamentals, weather, and storage into actionable view

class NatGasAnalyst:
    """LLM-powered natural gas market analyst."""
    
    def __init__(self, api_key: Optional[str] = None):
        self.llm = ChatAnthropic(
            model="claude-3-5-sonnet-20241022",
            api_key=api_key or os.environ.get("ANTHROPIC_API_KEY"),
            temperature=0.3
        )
        
    def generate_assessment(self,
                          fundamentals: NatGasFundamentals,
                          storage_balance: StorageBalance,
                          forecast: pd.DataFrame) -> str:
        """
        Generate market assessment for natural gas.
        
        Parameters
        ----------
        fundamentals : NatGasFundamentals
            Current fundamental data
        storage_balance : StorageBalance
            Storage balance analysis
        forecast : pd.DataFrame
            Forward storage trajectory
            
        Returns
        -------
        str
            Market assessment
        """
        context = f"""
NATURAL GAS MARKET ANALYSIS - {fundamentals.report_date}

STORAGE POSITION:
- Current: {storage_balance.current_level:.0f} Bcf
- vs 5-Year Avg: {storage_balance.vs_5yr_avg_pct:+.1f}%
- Weekly Change: {storage_balance.weekly_change:+.0f} Bcf
- Implied Weeks Supply: {storage_balance.implied_weeks_supply:.1f} weeks
- Season: {storage_balance.season}

SUPPLY/DEMAND:
Production: {fundamentals.supply_demand.production:.1f} Bcf/d
Total Consumption: {sum([fundamentals.supply_demand.residential_commercial, 
                        fundamentals.supply_demand.industrial,
                        fundamentals.supply_demand.power_generation]):.1f} Bcf/d
LNG Exports: {fundamentals.supply_demand.lng_exports:.1f} Bcf/d

FORECAST (12-week):
Projected End Level: {forecast['storage_level'].iloc[-1]:.0f} Bcf
Total Withdrawals: {forecast['storage_level'].iloc[0] - forecast['storage_level'].iloc[-1]:.0f} Bcf
End vs 5-Year Avg: {forecast['vs_avg_pct'].iloc[-1]:+.1f}%

QUALITATIVE FACTORS:
{chr(10).join('- ' + f for f in fundamentals.qualitative_factors)}
"""
        
        prompt = ChatPromptTemplate.from_messages([
            ("system", """You are an expert natural gas market analyst. Provide a 
comprehensive market assessment focusing on:

1. Storage adequacy for remainder of winter
2. Supply/demand balance
3. Weather sensitivity and risks
4. Price outlook (bullish/neutral/bearish)
5. Key risks and catalysts

Be specific about storage levels, withdrawal rates, and weather impacts."""),
            ("user", "{context}")
        ])
        
        chain = prompt | self.llm
        response = chain.invoke({"context": context})
        
        return response.content


# Generate assessment
analyst = NatGasAnalyst()
market_assessment = analyst.generate_assessment(
    gas_fundamentals, 
    storage_balance, 
    baseline_forecast
)

print("=== NATURAL GAS MARKET ASSESSMENT ===")
print(market_assessment)

## Exercise 4.2: Analyze a Different Weather Scenario

In [ ]:
# Exercise: Model an extreme weather scenario
#
# Task:
# 1. Create a "polar vortex" scenario (50% higher HDDs)
# 2. Forecast storage trajectory under this scenario
# 3. Calculate end-of-winter storage level
# 4. Generate LLM assessment of risks

# YOUR CODE HERE
# ---------------

polar_vortex_forecast = None  # Replace with your forecast

# Calculate key metrics
end_storage = None  # Final storage level
total_withdrawals = None  # Total Bcf withdrawn

# Generate assessment
extreme_assessment = None  # LLM assessment

In [ ]:
# SOLUTION

# Create polar vortex scenario (1.5x normal withdrawals)
polar_vortex_forecast = forecast_storage_trajectory(
    current_level=gas_fundamentals.storage.level,
    avg_withdrawal=abs(gas_fundamentals.storage.weekly_change),
    weather_adjustment=1.5
)

end_storage = polar_vortex_forecast['storage_level'].iloc[-1]
total_withdrawals = polar_vortex_forecast['storage_level'].iloc[0] - end_storage

print(f"Polar Vortex Scenario:")
print(f"End Storage: {end_storage:.0f} Bcf")
print(f"Total Withdrawals: {total_withdrawals:.0f} Bcf")
print(f"vs 5-Year Avg: {polar_vortex_forecast['vs_avg_pct'].iloc[-1]:+.1f}%")

# Visualize
plt.figure(figsize=(12, 6))
plt.plot(baseline_forecast['date'], baseline_forecast['storage_level'],
         label='Baseline', linewidth=2)
plt.plot(polar_vortex_forecast['date'], polar_vortex_forecast['storage_level'],
         label='Polar Vortex', linewidth=2, color='red')
plt.plot(baseline_forecast['date'], baseline_forecast['five_year_avg'],
         label='5-Year Average', linestyle='--', color='gray')
plt.fill_between(polar_vortex_forecast['date'],
                 baseline_forecast['storage_level'],
                 polar_vortex_forecast['storage_level'],
                 alpha=0.3, color='red', label='Additional Withdrawals')
plt.xlabel('Date')
plt.ylabel('Storage (Bcf)')
plt.title('Polar Vortex Impact on Storage')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Summary

### Key Takeaways

1. **Storage-Centric Analysis**: Natural gas fundamentals revolve around storage levels vs seasonal norms
2. **Weather Sensitivity**: HDD/CDD are critical demand drivers requiring continuous monitoring
3. **Seasonal Patterns**: Injection (Apr-Oct) and withdrawal (Nov-Mar) seasons drive market dynamics
4. **Forward Forecasting**: Storage trajectory forecasts assess supply adequacy
5. **Scenario Analysis**: Weather uncertainty creates wide range of outcomes

### What's Next

In **Module 4.3: Integrated Fundamental Model**, we'll:
- Combine crude oil and natural gas models
- Build cross-commodity relationships (oil production → associated gas)
- Create unified fundamental dashboards
- Generate portfolio-level fundamental views

### Additional Resources

- [EIA Natural Gas Weekly Update](https://www.eia.gov/naturalgas/weekly/)
- [NOAA Climate Prediction Center](https://www.cpc.ncep.noaa.gov/)
- "Natural Gas Trading" by Fletcher Sturm
- EIA Natural Gas Explained series

---

**Practice Exercise**: Track actual EIA storage reports for 4 weeks and compare forecasted vs actual storage changes. Analyze forecast errors and weather impacts.